# Experiment 01 — Transformer scaffold (Stage 1)

## 1. Research question

Can we stand up the **input stack** of DeepSeekFlashV4-Mini (tokenizer → embedding → RMSNorm → RoPE) with correct shapes, numerics, and a measurable tiny LM run on ≤2 GB / CPU?

## 2. Paper reference

- DeepSeek-V4.1-Flash tech report (`docs/DeepSeek_V41_Tech_Report.pdf`): Fig. 3 embedding entry; RMSNorm / RoPE discussion in architecture & KV-precision sections; Table 1 scale of V4-Flash (284B) vs what we **cannot** reproduce.
- RoPE: Su et al.; RMSNorm: Zhang & Sennrich (2019) — both used in DeepSeek stacks.

## 3. Mathematical explanation

**Embedding.** \(E \in \mathbb{R}^{V \times d}\), \(x_t = E_{z_t}\).

**RMSNorm.**
$$
\mathrm{RMS}(x)=\sqrt{\frac{1}{d}\sum_{i=1}^{d} x_i^2 + \varepsilon},\quad
y = \frac{x}{\mathrm{RMS}(x)} \odot \gamma
$$

**RoPE** (complex form on pairs):
$$
\tilde{q}_t = q_t \, e^{i t \theta},\qquad \theta_i = \mathrm{base}^{-2i/d}
$$


In [1]:
import sys
from pathlib import Path

PACKAGES = Path.cwd()
while not (PACKAGES / "deepseek_flash_v4_mini").exists():
    if PACKAGES == PACKAGES.parent:
        raise RuntimeError("Run this notebook from inside deepseek_flash_lab")
    PACKAGES = PACKAGES.parent
sys.path.insert(0, str(PACKAGES))
PKG = PACKAGES / "deepseek_flash_v4_mini"
import torch
from deepseek_flash_v4_mini.tokenizer import CharTokenizer
from deepseek_flash_v4_mini.model.embedding import TokenEmbedding
from deepseek_flash_v4_mini.model.normalization import RMSNorm
from deepseek_flash_v4_mini.model.rope import apply_rope, rope_freqs
print('stage-1 imports OK')


stage-1 imports OK


## 4. Tensor shapes

| Tensor | Shape |
|--------|-------|
| tokens | `[B, T]` |
| emb out | `[B, T, d]` |
| q/k for RoPE | `[B, H, T, d_h]` |
| RoPE freqs | `[T, d_h/2]` complex |


In [2]:
B, T, d, H, dh = 2, 8, 64, 4, 16
tok = CharTokenizer.from_text('hello world once upon a time')
emb = TokenEmbedding(tok.vocab_size, d)
norm = RMSNorm(d)
ids = torch.tensor([tok.encode('hello wo')[:T] for _ in range(B)])
x = emb(ids)
print('emb', tuple(x.shape))
print('norm', tuple(norm(x).shape))
q = torch.randn(B, H, T, dh)
print('rope', tuple(apply_rope(q).shape), 'freqs', tuple(rope_freqs(dh, T, q.device).shape))


emb (2, 8, 64)
norm (2, 8, 64)
rope (2, 4, 8, 16) freqs (8, 8)


## 5. Naive implementation

Complex multiply RoPE in `model/rope.py` (`apply_rope`).

## 6. Optimized implementation

Deferred (fused RoPE-attention kernels are production-only). Stage-1 keeps the readable complex form.

## 7. Correctness test


In [3]:
import subprocess, sys
r = subprocess.run([sys.executable, str(PKG / 'tests' / 'test_rope.py')], capture_output=True, text=True)
print(r.stdout)
print(r.stderr)
assert r.returncode == 0


test_rope: ALL PASSED




## 8. Training experiment

Run `training/stage01_train.py` (Level-0 char corpus; temporary inline attention marked `[EXPERIMENTAL]`).


In [4]:
import subprocess, sys, json
r = subprocess.run([sys.executable, str(PKG / 'training' / 'stage01_train.py')], capture_output=True, text=True)
print(r.stdout[-2000:] if len(r.stdout)>2000 else r.stdout)
print(r.stderr)
assert r.returncode == 0
metrics = json.loads((PACKAGES.parent / 'results' / 'stage01' / 'metrics.json').read_text())
metrics


=== Stage-1 memory print ===
device=cpu
parameters=100,480
{
  "param_mb": 0.383,
  "optimizer_mb_est": 0.767,
  "activation_mb_est": 0.094,
  "total_est_mb": 1.244
}
seq=48 batch=4 steps=100
=== Stage-1 results ===
{
  "model": "DeepSeekFlashV4-Mini",
  "experiment": "stage01_rope_baseline",
  "fidelity": {
    "tokenizer": "[SIMPLIFIED]",
    "embedding": "[FAITHFUL]",
    "rmsnorm": "[FAITHFUL]",
    "rope": "[FAITHFUL]",
    "attention_in_train_loop": "[EXPERIMENTAL] temporary inline"
  },
  "parameters": 100480,
  "dataset": "level0_char_corpus",
  "dataset_chars": 401,
  "dataset_tokens": 401,
  "vocab_size": 29,
  "sequence_length": 48,
  "batch_size": 4,
  "steps": 100,
  "train_loss": 1.7628531455993652,
  "validation_loss": 1.602990746498108,
  "perplexity_val": 4.967867862232226,
  "device": "cpu",
  "peak_gpu_memory_mb": 0.0,
  "peak_cpu_memory_mb": 781.34,
  "training_time_s": 0.293,
  "tokens_per_second": 65580.51,
  "memory_estimates_mb": {
    "param_mb": 0.383,
    "op

{'model': 'DeepSeekFlashV4-Mini',
 'experiment': 'stage01_rope_baseline',
 'fidelity': {'tokenizer': '[SIMPLIFIED]',
  'embedding': '[FAITHFUL]',
  'rmsnorm': '[FAITHFUL]',
  'rope': '[FAITHFUL]',
  'attention_in_train_loop': '[EXPERIMENTAL] temporary inline'},
 'parameters': 100480,
 'dataset': 'level0_char_corpus',
 'dataset_chars': 401,
 'dataset_tokens': 401,
 'vocab_size': 29,
 'sequence_length': 48,
 'batch_size': 4,
 'steps': 100,
 'train_loss': 1.7628531455993652,
 'validation_loss': 1.602990746498108,
 'perplexity_val': 4.967867862232226,
 'device': 'cpu',
 'peak_gpu_memory_mb': 0.0,
 'peak_cpu_memory_mb': 781.34,
 'training_time_s': 0.293,
 'tokens_per_second': 65580.51,
 'memory_estimates_mb': {'param_mb': 0.383,
  'optimizer_mb_est': 0.767,
  'activation_mb_est': 0.094,
  'total_est_mb': 1.244}}

## 9. Benchmark / 10. Result / 11. Interpretation

See printed `metrics.json`. Expect train/val loss to drop on the tiny repeated corpus; absolute numbers are **not** comparable to Flash.

## 12. Production difference

Cannot reproduce 284B parameters, BPE vocab, CSA–HCA, MoE at production width, or cluster KV replay on a 2 GB GPU. Stage-1 only verifies the **input + position** primitives.
